# GitHub Projects Data Analytics Pipeline

Level 3 Project 2

## Task 1: Data Collection & Preparation

### Step 1: Collect data from the GitHub API

In [ ]:
import requests
import pandas as pd

In [ ]:
# search endpoint for machine learning repos, sorted by stars
api_url = "https://api.github.com/search/repositories?q=machine+learning&sort=stars&order=desc&per_page=100"

In [ ]:
response = requests.get(api_url)
response.raise_for_status()

data = response.json()

In [ ]:
print("Total repos found:", data["total_count"])
print("Repos returned:", len(data["items"]))

### Step 2: Create the DataFrame and explore it

In [ ]:
# flatten the nested json into a table
df = pd.json_normalize(data["items"])
df.shape

In [ ]:
df.info()

In [ ]:
df.head()

### Step 3: Select the required columns

In [ ]:
columns_needed = [
    "name", "full_name", "owner.login", "html_url", "description",
    "language", "stargazers_count", "forks_count", "open_issues_count",
    "created_at", "updated_at", "license.name"
]

df = df[columns_needed].copy()

In [ ]:
# rename nested columns to something simpler
df = df.rename(columns={
    "owner.login": "owner",
    "license.name": "license"
})

df.head()

### Step 4: Handle missing values

In [ ]:
df.isnull().sum()

In [ ]:
# fill missing values instead of dropping rows
df["description"] = df["description"].fillna("No description")
df["license"] = df["license"].fillna("No license")
df["language"] = df["language"].fillna("Unknown")

In [ ]:
# drop rows only if the name is missing
df = df.dropna(subset=["name", "full_name"])

### Step 5: Handle duplicate records

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=["full_name"])
after = len(df)

print("Duplicates removed:", before - after)

### Step 6: Prepare the dataset for SQL

In [ ]:
# convert date columns to datetime
df["created_at"] = pd.to_datetime(df["created_at"])
df["updated_at"] = pd.to_datetime(df["updated_at"])

In [ ]:
# rename columns to simpler SQL friendly names
df = df.rename(columns={
    "stargazers_count": "stars",
    "forks_count": "forks",
    "open_issues_count": "open_issues"
})

df.dtypes

### Step 7: Save and verify the dataset

In [ ]:
df.to_csv("github_projects.csv", index=False)

In [ ]:
# reload the file to check it saved correctly
check = pd.read_csv("github_projects.csv")
print("Saved correctly:", check.shape == df.shape)
check.head()

## Task 2: Store & Analyse the Data

### Step 8: Create the SQLite database

In [ ]:
import sqlite3

conn = sqlite3.connect("github_projects.db")

In [ ]:
df.to_sql("Repositories", conn, if_exists="replace", index=False)

In [ ]:
# check the table has data in it
pd.read_sql("SELECT * FROM Repositories LIMIT 5", conn)

### Step 9: SQL analysis

Filtering and searching

In [ ]:
pd.read_sql("""
    SELECT name, owner, stars
    FROM Repositories
    WHERE language = 'Python' AND stars > 5000
""", conn)

Logical operators

In [ ]:
pd.read_sql("""
    SELECT name, language, stars
    FROM Repositories
    WHERE (language = 'Python' OR language = 'Jupyter Notebook')
    AND NOT license = 'No license'
""", conn)

Sorting and limiting

In [ ]:
pd.read_sql("""
    SELECT name, stars
    FROM Repositories
    ORDER BY stars DESC
    LIMIT 10
""", conn)

Aggregate functions

In [ ]:
pd.read_sql("""
    SELECT COUNT(*) AS total_repos, AVG(stars) AS avg_stars, MAX(stars) AS max_stars
    FROM Repositories
""", conn)

Grouping

In [ ]:
pd.read_sql("""
    SELECT language, COUNT(*) AS repo_count, AVG(stars) AS avg_stars
    FROM Repositories
    GROUP BY language
    HAVING COUNT(*) > 1
    ORDER BY avg_stars DESC
""", conn)

### Step 10: Create visualizations

In [ ]:
import matplotlib.pyplot as plt

Chart 1: Top 10 repos by stars

In [ ]:
top10 = pd.read_sql("""
    SELECT name, stars FROM Repositories ORDER BY stars DESC LIMIT 10
""", conn)

In [ ]:
# shorten long repo names so the chart is readable
short_names = []

for name in top10["name"]:
    if len(name) > 15:
        name = name[:15] + "..."
    short_names.append(name)

top10["short_name"] = short_names

In [ ]:
plt.figure(figsize=(9, 6))
plt.barh(top10["short_name"], top10["stars"], color="steelblue")
plt.xlabel("Stars")
plt.title("Top 10 Machine Learning Repos by Stars")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("top10_stars.png")
plt.show()

Chart 2: Repos by language

In [ ]:
lang_counts = pd.read_sql("""
    SELECT language, COUNT(*) AS count FROM Repositories GROUP BY language
""", conn)

lang_counts = lang_counts.sort_values("count", ascending=False)

In [ ]:
# combine small language groups into "Other" so the pie chart is readable
top_langs = lang_counts.head(5).copy()

other_count = 0
for count in lang_counts["count"][5:]:
    other_count = other_count + count

if other_count > 0:
    other_row = pd.DataFrame({"language": ["Other"], "count": [other_count]})
    top_langs = pd.concat([top_langs, other_row], ignore_index=True)

In [ ]:
plt.figure(figsize=(7, 7))
plt.pie(top_langs["count"], labels=top_langs["language"], autopct="%1.1f%%", startangle=90)
plt.title("Repositories by Language")
plt.tight_layout()
plt.savefig("language_pie.png")
plt.show()

Chart 3: Repos created by year

In [ ]:
df["created_year"] = df["created_at"].dt.year
year_counts = df.groupby("created_year").size().reset_index(name="count")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(year_counts["created_year"], year_counts["count"], marker="o")
plt.xlabel("Year")
plt.ylabel("Repos Created")
plt.title("Repository Creation Trend by Year")
plt.xticks(year_counts["created_year"], rotation=45)
plt.tight_layout()
plt.savefig("creation_trend.png")
plt.show()

### Step 11: Interpret results

- The top 10 chart shows a small number of repositories have far more stars than the rest.
- Python is the most common language in the dataset, which fits with machine learning being a Python-heavy field.
- The creation trend shows most repositories were created in recent years, matching the growth of interest in machine learning.

(Replace these notes with what the actual output shows after running the notebook.)

## Task 3: Version Control & Project Ethics

### Git and GitHub

Run these commands in a terminal, in the project folder:

```bash
git init
git status

git add .
git commit -m "Initial commit: collected and cleaned data"

git add .
git commit -m "Added SQL analysis and charts"

git remote add origin https://github.com/YOUR_USERNAME/github-ml-projects-analytics.git
git branch -M main
git push -u origin main
```

Take a screenshot of the commit history and the pushed repository as evidence.

### Ethics Reflection

**Question 1: Why is it important to verify data collected from public APIs?**
Public APIs can return data that is incomplete, outdated, or incorrect. Verifying the data makes sure the analysis is based on accurate information and not on errors that came from the source itself.

**Question 2: Why should data analysts document the source of their data?**
Documenting the source lets other people check where the data came from, repeat the same collection process, and trust that the analysis can be verified. It also makes the work more transparent.

**Question 3: How can missing or inaccurate data affect data analysis and decision-making?**
Missing or inaccurate data can lead to wrong conclusions and charts that do not represent reality. If decisions are made based on this data, they may be based on a false picture of the situation, which can lead to poor outcomes.

GitHub Repository: https://github.com/mohaM4e/lv3_deci_project.git